# Approach 2b — Per-run IPA, then average across runs

**Pipeline:** For each of the 100 per-run fits in `per_run_fits_p_{p}_bs_{bs}.csv`, compute IPA using that run's own `(A, B, n)` and its own BN grid → average and std over runs.

**Only this approach has a meaningful STD** (across-run variability).  STD columns in the other three summaries are NaN by design.

In [12]:
# === Cell 1 — Config, imports, helpers ===
import os, glob, re
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Paths
BASE_DIR = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
FIT_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\Fitting_IPA_curves_data_I"
OUT_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_2b"
INTERMEDIATE_DIR = os.path.join(OUT_DIR, "intermediate")
os.makedirs(INTERMEDIATE_DIR, exist_ok=True)

BATCH_SIZES = [64, 1024, 60000]
CE_o = np.log(10)   # max CE for 10-class problem, ~2.302585

# Auto-detect pruning percentages from prune_layers_ALL/p-percentage_*/
p_dirs = glob.glob(os.path.join(BASE_DIR, "p-percentage_*"))
PRUNING_LEVELS = sorted([
    float(re.search(r"p-percentage_([\d.]+)", d).group(1))
    for d in p_dirs
])
print(f"Found {len(PRUNING_LEVELS)} pruning percentages: {PRUNING_LEVELS}")
print(f"CE_o = ln(10) = {CE_o:.6f}")

# --- IPA from fit (single source of truth) ---
# CE_L is the physically meaningful learning threshold.
#   CE_L = CE_o - 0.9 * (CE_o - A)
#   IPA  = abs(CE_o - CE_L) / learn_BN   where learn_BN = ceil of analytical BN crossing.
# We intentionally do NOT simplify to 0.9*(CE_o - A)/learn_BN — CE_L stays a first-class variable.
# learn_BN is always from the smooth fitted function (analytical inverse), never from raw grid points.
def compute_ipa_from_fit(A, B, n):
    CE_L = CE_o - 0.9 * (CE_o - A)
    denom = CE_L - A
    if denom <= 0 or n <= 0 or B <= 0:
        return {"CE_L": CE_L, "learn_BN": np.nan, "IPA": np.nan, "fitted_at_learn_BN": np.nan}
    BN_analytic = (B / denom) ** (1.0 / n) - 1.0
    if not np.isfinite(BN_analytic) or BN_analytic <= 0:
        return {"CE_L": CE_L, "learn_BN": np.nan, "IPA": np.nan, "fitted_at_learn_BN": np.nan}
    learn_BN = float(np.ceil(BN_analytic))
    if learn_BN == 0:
        return {"CE_L": CE_L, "learn_BN": 0.0, "IPA": np.nan, "fitted_at_learn_BN": float(A + B / (1.0 ** n))}
    fitted_at = float(A + B / ((learn_BN + 1) ** n))
    IPA = abs(CE_o - CE_L) / learn_BN
    return {"CE_L": CE_L, "learn_BN": learn_BN, "IPA": IPA, "fitted_at_learn_BN": fitted_at}
print("Cell 1 ready.")

Found 19 pruning percentages: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.82, 0.84, 0.86, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 1.0]
CE_o = ln(10) = 2.302585
Cell 1 ready.


In [13]:
# === Cell 2 — Approach 2b: per-run IPA, then average across runs ===
# For each (P%, BS): groupby Run in per_run_fits_p_{p}_bs_{bs}.csv,
# extract (A, B, n), compute one IPA per run from the analytical fitted-curve inverse,
# then aggregate (mean, std, n_valid) across runs.
inter_by_bs = {}

for bs in BATCH_SIZES:
    print("\n" + "=" * 70)
    print(f"  Approach 2b — Batch size {bs}")
    print("=" * 70)
    rows = []
    for p in PRUNING_LEVELS:
        per_run_csv = os.path.join(FIT_DIR, f"BS_{bs}", f"per_run_fits_p_{p}_bs_{bs}.csv")
        if not os.path.exists(per_run_csv):
            print(f"  [SKIP] P%={p*100:5.1f}%  — missing {per_run_csv}")
            continue
        df = pd.read_csv(per_run_csv)
        df.columns = df.columns.str.strip()

        run_rows = []
        for run_name, sub in df.groupby("Run"):
            A = float(sub["A"].iloc[0])
            B = float(sub["B"].iloc[0])
            n = float(sub["n"].iloc[0])
            ipa = compute_ipa_from_fit(A, B, n)
            run_rows.append({
                "Run": run_name, "A": A, "B": B, "n": n,
                "CE_o": CE_o, "CE_L": ipa["CE_L"],
                "learn_BN": ipa["learn_BN"], "fitted_at_learn_BN": ipa["fitted_at_learn_BN"],
                "IPA": ipa["IPA"],
            })

        if not run_rows:
            print(f"  [SKIP] P%={p*100:5.1f}%  — no runs in file")
            continue

        run_df = pd.DataFrame(run_rows)
        per_run_out = os.path.join(INTERMEDIATE_DIR,
                                    f"approach_2b_per_run_p_{p}_bs_{bs}.csv")
        run_df.to_csv(per_run_out, index=False)

        ipa_vals = run_df["IPA"].dropna().values
        n_valid  = len(ipa_vals)
        ipa_mean = float(np.mean(ipa_vals)) if n_valid >= 1 else np.nan
        ipa_std  = float(np.std(ipa_vals, ddof=1)) if n_valid >= 2 else np.nan

        print(f"  P%={p*100:5.1f}%  runs={len(run_df)}  valid={n_valid}  "
              f"IPA_mean={ipa_mean}  IPA_std={ipa_std}")

        rows.append({
            "P%": p * 100,
            "n_runs": len(run_df), "n_valid": n_valid,
            "IPA_mean": ipa_mean, "IPA_std": ipa_std,
        })

    if rows:
        bs_df = pd.DataFrame(rows)
        inter_path = os.path.join(INTERMEDIATE_DIR, f"approach_2b_summary_bs_{bs}.csv")
        bs_df.to_csv(inter_path, index=False)
        print(f"  Saved: {inter_path}")
        inter_by_bs[bs] = bs_df

print("\n[Cell 2 done]")


  Approach 2b — Batch size 64
  P%=  0.0%  runs=100  valid=100  IPA_mean=0.049649164612316804  IPA_std=0.04011741233357213
  P%= 10.0%  runs=100  valid=100  IPA_mean=0.07689063262308646  IPA_std=0.1932672509688931
  P%= 20.0%  runs=100  valid=100  IPA_mean=0.04553269953001911  IPA_std=0.05382360262896967
  P%= 30.0%  runs=100  valid=100  IPA_mean=0.06583483875937388  IPA_std=0.18236064325464707
  P%= 40.0%  runs=100  valid=100  IPA_mean=0.04189110188269358  IPA_std=0.05514095326106333
  P%= 50.0%  runs=100  valid=100  IPA_mean=0.032319321732771106  IPA_std=0.0524189192853983
  P%= 60.0%  runs=100  valid=100  IPA_mean=0.016438448077437856  IPA_std=0.0022344574817288825
  P%= 70.0%  runs=100  valid=100  IPA_mean=0.012730921918962775  IPA_std=0.007913570392237258
  P%= 80.0%  runs=100  valid=100  IPA_mean=0.007068215275836767  IPA_std=0.0010840298573543887
  P%= 82.0%  runs=63  valid=63  IPA_mean=0.006050156741130639  IPA_std=0.0008373712135218935
  P%= 84.0%  runs=63  valid=63  IPA_mean

In [14]:
# === Cell 3 — Build wide summary CSV for Approach 2b ===
# Schema: P%, IPA_Avg_64, STD_64, IPA_Avg_1024, STD_1024, IPA_Avg_60000, STD_60000
# STD columns blank (NaN) for single-curve approaches; populated only by 2b.
summary_rows = []
for p in PRUNING_LEVELS:
    row = {"P%": p * 100}
    for bs in BATCH_SIZES:
        df = inter_by_bs.get(bs)
        if df is None:
            mean_val, std_val = np.nan, np.nan
        else:
            sub = df[df["P%"] == p * 100]
            mean_val = float(sub["IPA"].iloc[0]) if (not sub.empty and "IPA" in sub.columns) else (
                       float(sub["IPA_mean"].iloc[0]) if (not sub.empty and "IPA_mean" in sub.columns) else np.nan)
        std_val  = float(sub["IPA_std"].iloc[0]) if not sub.empty and "IPA_std" in sub.columns else np.nan
        row[f"IPA_Avg_{bs}"] = mean_val
        row[f"STD_{bs}"]     = std_val
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows, columns=[
    "P%", "IPA_Avg_64", "STD_64", "IPA_Avg_1024", "STD_1024", "IPA_Avg_60000", "STD_60000"
])
out_csv = os.path.join(OUT_DIR, "ipa_summary_approach_2b.csv")
summary_df.to_csv(out_csv, index=False)
print(f"\nFinal summary written: {out_csv}")
print(summary_df.to_string(index=False))



Final summary written: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_2b\ipa_summary_approach_2b.csv
   P%  IPA_Avg_64   STD_64  IPA_Avg_1024  STD_1024  IPA_Avg_60000  STD_60000
  0.0    0.049649 0.040117      0.084372  0.180353       0.071620   0.004521
 10.0    0.076891 0.193267      0.077945  0.109735       0.064588   0.002873
 20.0    0.045533 0.053824      0.075046  0.100425       0.057657   0.001774
 30.0    0.065835 0.182361      0.061968  0.075337       0.050828   0.002192
 40.0    0.041891 0.055141      0.094923  0.212323       0.044187   0.001836
 50.0    0.032319 0.052419      0.057921  0.113827       0.038745   0.015918
 60.0    0.016438 0.002234      0.067488  0.125773       0.033184   0.033084
 70.0    0.012731 0.007914      0.065375  0.142804       0.051422   0.088259
 80.0    0.007068 0.001084      0.031251  0.169091       0.041951   0.076307
 82.0    0.006050 0.000837      0.011218  0.001660       0.021447   0.038843
 84.0    0.0

In [17]:
# === Cell 4 — Plot IPA vs P% ===
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

TAG   = "2b"
TITLE = "Approach 2b — Per-run IPA, averaged across runs"
BS_COLOR = {64: "#1f77b4", 1024: "#d62728", 60000: "#2ca02c"}

plt.rcParams.update({"font.size": 14})
fig, ax = plt.subplots(figsize=(10, 6))

for bs in BATCH_SIZES:
    mean_col = f"IPA_Avg_{bs}"
    std_col  = f"STD_{bs}"
    sub = summary_df.dropna(subset=[mean_col])
    if sub.empty:
        continue
    ax.errorbar(sub["P%"].values, sub[mean_col].values, yerr=sub[std_col].values,
                label=f"BS={bs}", color=BS_COLOR[bs], marker="o", markersize=6,
                linewidth=2, capsize=3)

ax.set_xlabel("Pruning Percentage (%)")
ax.set_ylabel("IPA")
# ax.set_yscale("log")
ax.set_title(TITLE, fontsize=14)
ax.grid(True, which="both", alpha=0.3)
ax.legend(frameon=False)

out_png = os.path.join(OUT_DIR, f"ipa_plot_approach_{TAG}.png")
plt.tight_layout()
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_png}")

Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_2b\ipa_plot_approach_2b.png
